# Experiment 2: FIND-S Algorithm

## Overview
The FIND-S algorithm finds the most specific hypothesis that is consistent with all positive examples in the training set. It assumes:
- Instances are described by conjunctions of attributes
- The hypothesis space is based on conjunctions of constraints
- Learning proceeds by finding the most specific hypothesis

## Algorithm Steps
1. Initialize hypothesis to the most specific hypothesis (all attributes set to ?)
2. For each positive example:
   - For each attribute in the hypothesis:
     - If the constraint is not satisfied by the example, specialize it
3. Return the final hypothesis

In [ ]:
import pandas as pd
import numpy as np
from itertools import product

print("FIND-S Algorithm Implementation")
print("=" * 50)

## Sample Dataset Creation

In [ ]:
# Create sample dataset - Binary classification for enjoying sports
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny', 'Cloudy'],
    'Temperature': ['Warm', 'Warm', 'Cold', 'Warm', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High', 'Normal'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Strong'],
    'Water': ['Warm', 'Cool', 'Warm', 'Cool', 'Warm'],
    'Forecast': ['Same', 'Same', 'Change', 'Change', 'Same'],
    'EnjoySports': ['Yes', 'Yes', 'No', 'Yes', 'Yes']
}

df = pd.DataFrame(data)
print("\nTraining Dataset:")
print(df)
print(f"\nPositive Examples (EnjoySports = Yes): {len(df[df['EnjoySports'] == 'Yes'])}")
print(f"Negative Examples (EnjoySports = No): {len(df[df['EnjoySports'] == 'No'])}")

## FIND-S Algorithm Implementation

In [ ]:
def find_s_algorithm(dataset, target_column):
    """
    Implement FIND-S algorithm to find the most specific hypothesis
    
    Parameters:
    -----------
    dataset : pd.DataFrame
        Training dataset
    target_column : str
        Name of the target column
    
    Returns:
    --------
    hypothesis : dict
        The most specific hypothesis
    """
    # Get feature names (exclude target column)
    attributes = [col for col in dataset.columns if col != target_column]
    
    # Initialize hypothesis with '?' (most general)
    hypothesis = ['?'] * len(attributes)
    
    print("\n" + "="*60)
    print("FIND-S ALGORITHM EXECUTION")
    print("="*60)
    print(f"\nInitial hypothesis (most general): {hypothesis}")
    print(f"Attributes: {attributes}")
    
    # Get positive examples only
    positive_examples = dataset[dataset[target_column] == 'Yes']
    
    print(f"\nProcessing {len(positive_examples)} positive examples...\n")
    
    # Process each positive example
    for idx, (example_idx, example) in enumerate(positive_examples.iterrows(), 1):
        print(f"Example {idx}: {dict(example[attributes])}")
        
        # First positive example: copy all values
        if hypothesis == ['?'] * len(attributes):
            hypothesis = list(example[attributes].values)
            print(f"  → Initialize hypothesis to: {hypothesis}")
        else:
            # Check each attribute
            for attr_idx, attribute in enumerate(attributes):
                example_value = example[attribute]
                
                # If hypothesis value is '?', it's already general enough
                if hypothesis[attr_idx] == '?':
                    print(f"  → Attribute '{attribute}': '?' is general - no change")
                # If hypothesis value matches example value, keep it
                elif hypothesis[attr_idx] == example_value:
                    print(f"  → Attribute '{attribute}': '{hypothesis[attr_idx]}' = '{example_value}' - keep it")
                # If they don't match, generalize to '?'
                else:
                    hypothesis[attr_idx] = '?'
                    print(f"  → Attribute '{attribute}': '{hypothesis[attr_idx]} != {example_value}' - generalize to '?'")
        
        print(f"  → Current hypothesis: {hypothesis}\n")
    
    # Create dictionary for hypothesis
    final_hypothesis = dict(zip(attributes, hypothesis))
    
    return final_hypothesis, attributes, hypothesis

# Run FIND-S algorithm
hypothesis_dict, attributes, hypothesis = find_s_algorithm(df, 'EnjoySports')

## Results

In [ ]:
# Display final hypothesis
print("\n" + "="*60)
print("FINAL HYPOTHESIS")
print("="*60)
print("\nMost Specific Hypothesis:")
for attr, val in hypothesis_dict.items():
    print(f"  {attr}: {val}")

# Create a more readable representation
print("\nHypothesis in readable form:")
conditions = []
for attr, val in hypothesis_dict.items():
    if val != '?':
        conditions.append(f"{attr} = {val}")

if conditions:
    print("If ( " + " AND ".join(conditions) + " ) then EnjoySports = Yes")
else:
    print("< ?, ?, ?, ?, ?, ? > - Most general hypothesis (matches everything)")

## Verification

In [ ]:
def matches_hypothesis(example, hypothesis_dict):
    """
    Check if an example matches the hypothesis
    """
    for attr, hyp_val in hypothesis_dict.items():
        if hyp_val != '?':
            if example[attr] != hyp_val:
                return False
    return True

# Verify against all examples
print("\n" + "="*60)
print("VERIFICATION AGAINST TRAINING DATA")
print("="*60)

correct = 0
total = 0

print("\nPositive Examples (Should match hypothesis):")
for idx, (example_idx, example) in enumerate(df[df['EnjoySports'] == 'Yes'].iterrows(), 1):
    matches = matches_hypothesis(example, hypothesis_dict)
    status = "✓ Matches" if matches else "✗ Does not match"
    print(f"  {idx}. {dict(example[attributes])} - {status}")
    if matches:
        correct += 1
    total += 1

print("\nNegative Examples (Should NOT match hypothesis):")
for idx, (example_idx, example) in enumerate(df[df['EnjoySports'] == 'No'].iterrows(), 1):
    matches = matches_hypothesis(example, hypothesis_dict)
    status = "✓ Correctly excluded" if not matches else "✗ Incorrectly matches"
    print(f"  {idx}. {dict(example[attributes])} - {status}")
    if not matches:
        correct += 1
    total += 1

print(f"\nAccuracy: {correct}/{total} = {100*correct/total:.2f}%")

## Summary

**FIND-S Algorithm Key Points:**
- Finds the most specific hypothesis consistent with positive examples
- Ignores negative examples
- Produces one hypothesis (the most specific)
- Not suitable for problems with inconsistent data
- Cannot produce multiple hypotheses
- Limited to conjunctive hypotheses

**Advantages:**
- Simple and efficient
- Guaranteed to find most specific hypothesis

**Disadvantages:**
- Cannot handle noisy data
- No mechanism for handling inconsistencies
- Doesn't tell us if we've found the best hypothesis

## Viva Questions

### Basic Concepts
1. **What is the FIND-S algorithm used for?**
   - Finds the most specific hypothesis that is consistent with all positive examples
   - Part of concept learning from positive examples

2. **Explain the term "most specific hypothesis".**
   - Hypothesis that rules out the smallest set of instances
   - Uses minimum generalization necessary to cover positive examples

3. **What are the assumptions made by FIND-S?**
   - Target function is conjunctive (AND of attributes)
   - Instances described by conjunctions of attributes
   - Attributes have finite discrete values

4. **What is the role of '?' in FIND-S?**
   - '?' represents "any value" (most general for that attribute)
   - Opposite of specific attribute values

### Algorithm Questions
5. **Describe the FIND-S algorithm step by step.**
   - Initialize hypothesis to most specific (e.g., all values set to '?')
   - For each positive example: if attribute value doesn't match hypothesis, generalize to '?'
   - Return final hypothesis

6. **Why does FIND-S ignore negative examples?**
   - Algorithm is designed to find hypothesis consistent with positive examples only
   - Negative examples don't directly guide the learning process

7. **Can FIND-S produce multiple hypotheses?**
   - No, it produces only one hypothesis (the most specific)
   - Returns a single conjunctive hypothesis

8. **What happens when FIND-S encounters inconsistent data?**
   - May produce overly general or incorrect hypothesis
   - Cannot handle contradictory examples well

### Limitations & Comparison
9. **What are the main limitations of FIND-S?**
   - Cannot handle noisy or inconsistent data
   - Provides only one hypothesis
   - No mechanism to check if found hypothesis is the best
   - Limited to conjunctive hypotheses

10. **How does FIND-S differ from Candidate-Elimination algorithm?**
    - FIND-S finds one most specific hypothesis
    - Candidate-Elimination maintains all consistent hypotheses (version space)
    - CE is more robust to version space updates

11. **When is FIND-S preferred over other learning algorithms?**
    - When data is consistent and positive examples are reliable
    - For problems with clear conjunctive patterns
    - Educational purpose to understand concept learning

12. **Can FIND-S be extended for disjunctive hypotheses?**
    - No, it's fundamentally designed for conjunctive (AND) patterns
    - Would need different algorithm for OR patterns